# 04. Vector Search with LanceDB (Real Titles)

Notebook này thực hiện thử nghiệm lưu trữ và truy vấn Vector Embeddings sử dụng dữ liệu tên phim thực tế từ bộ dữ liệu MovieLens 25M.

In [ ]:
import lancedb
import numpy as np
import pandas as pd
import os

## 1. Load Tên phim thực tế từ Dataset

In [ ]:
MOVIES_PATH = "../dataset/ml-25m/movies.csv"
df_movies_raw = pd.read_csv(MOVIES_PATH)

# Lấy mẫu 2000 bộ phim đầu tiên để demo
df_sample = df_movies_raw.head(2000).copy()

# Giả lập Vector Embedding 32 chiều cho mỗi bộ phim (Đầu ra giả định từ Item Tower)
df_sample["vector"] = [np.random.randn(32).astype(np.float32).tolist() for _ in range(len(df_sample))]

print(f"Đã nạp {len(df_sample)} phim thực tế kèm vector giả lập.")
df_sample[['movieId', 'title', 'genres']].head()

## 2. Khởi tạo LanceDB và Lưu trữ

In [ ]:
db = lancedb.connect("./tmp_lancedb")

# Tạo bảng movies với dữ liệu thật
table = db.create_table("movies_real", data=df_sample, mode="overwrite")
print(f"Đã tạo bảng 'movies_real' thành công.")

## 3. Truy vấn gợi ý phim thực tế

In [ ]:
# Giả lập một Vector sở thích của người dùng (Đầu ra từ User Tower)
query_user_vector = np.random.randn(32).astype(np.float32)

print("--- Đang tìm kiếm Top 5 phim phù hợp nhất với sở thích người dùng ---")
results = table.search(query_user_vector).limit(5).to_pandas()

# Hiển thị kết quả với tên phim thật
results[['title', 'genres', '_distance']]

## 4. Mô phỏng Semantic Search (Tìm kiếm theo ý nghĩa)

Trong thực tế, khi người dùng gõ: "Phim hoạt hình cho trẻ em", chúng ta dùng mô hình NLP biến câu này thành vector và search trong bảng này.

In [ ]:
query_text = "Phim hành động kịch tính của thập niên 90"
print(f"Tìm kiếm vector cho: '{query_text}'")

# Search kết quả
semantic_results = table.search(np.random.randn(32)).limit(3).to_pandas()
semantic_results[['title', 'genres']]